<a href="https://colab.research.google.com/github/mugalan/data-analysis-tool/blob/main/data_analysis_cookbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overview

A robust Python toolkit designed for data cleaning, exploration, and interactive visualization within **Google Colab** environments. This tool automates common preprocessing tasks such as data sanitization, missing value imputation, outlier detection, and advanced statistical association mapping.

## Features

- **Intelligent Data Loading**: Automatically handles common null strings (e.g., `'?'`, `'N/A'`, `'NULL'`) and attempts auto-conversion of columns to correct numeric types.
- **Comprehensive Inspection**: Quickly view data dimensions, column type breakdowns, and statistical summaries for both numerical and categorical data.
- **Automated Cleaning**:
  - Identify and impute missing values using mean, median, mode, or constant strategies.
  - Remove exact duplicate rows and specific outliers using IQR logic.
  - Delete specific rows and columns by index/name.
- **Advanced Scaling & Encoding**:
  - *Numeric*: Min-Max, Standard (Z-score), and Robust scaling.
  - *Categorical*: One-Hot, Ordinal, and Uniform encoding.
- **Interactive Visualizations**: Powered by Plotly, including horizontal violin plots, scatter plots, histograms, and grouped bar charts.
- **Deep Statistical Insights**:
  - Pearson Correlation heatmaps for numeric data.
  - Cramér's V heatmaps for categorical associations.
  - Unified Association Heatmaps combining Numeric and Categorical data.

# Python Package Installation

In [ ]:
import json, csv
import pandas as pd
# import numpy as np
# import scipy.stats
# import seaborn as sns
# import matplotlib.pyplot as plt
# from IPython.display import display, Image, HTML
# from pydantic import BaseModel, ValidationError, field_validator
# from typing import Optional


In [ ]:
!pip install "git+https://github.com/mugalan/data-analysis-tool.git"


In [ ]:
from data_analysis import DataInspector, PlottingMethods


# Quick Start Example

In [ ]:
# Initialize the inspector
inspector = DataInspector()


## Data Import

Choose **one** of the data-loading methods below. Each method sets `inspector.df` — run only the block that suits your use case.

In [ ]:
# ── Option 1: Interactive file upload (Colab file-picker) ──
# Opens a browser button to upload any CSV from your computer.
inspector.upload_data()


In [ ]:
# ── Option 2: Load directly from a public URL ──
# Direct URL to the Titanic CSV on GitHub
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url)
print(f"✅ Loaded {len(inspector.df)} rows from URL")


In [ ]:
# ── Option 3: UCI ML Repository ──
!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo

# Fetch dataset (id=597 → Garment-worker productivity)
productivity_prediction_of_garment_employees = fetch_ucirepo(id=597)

# Data (as pandas DataFrames)
X = productivity_prediction_of_garment_employees.data.features
y = productivity_prediction_of_garment_employees.data.targets

# Metadata
print(productivity_prediction_of_garment_employees.metadata)
print(productivity_prediction_of_garment_employees.variables)


In [ ]:
# Assign UCI dataset to inspector
inspector.df = X


In [ ]:
# ── Option 4: Kaggle dataset ──
# Requires Kaggle credentials configured in Colab (Settings → Secrets → KAGGLE_KEY / KAGGLE_USERNAME)
!pip install -q kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ziya07/fault-diagnosis-dataset-for-new-energy-vehicles")
print("Path to dataset files:", path)


In [ ]:
import os
print(f'Listing contents of: {path}')
!ls {path}
df2 = pd.read_csv(path + "/NEV_fault_training_dataset.csv")
inspector.df = df2


## Initial Data Inspection and Cleaning

In [ ]:
inspector.get_summary()


In [ ]:
# Delete columns by name — pass a list of column names to drop.
# Run with no arguments first to see available columns.
inspector.delete_columns()
# Example: inspector.delete_columns(['PassengerId', 'Cabin', 'Ticket'])


In [ ]:
inspector.column_details()


In [ ]:
inspector.get_categorical_summary()


In [ ]:
# Show rows with missing values.
# Pass column='ColumnName' to also see positional missing-value masks for that column.
inspector.show_missing_data()
# Example: inspector.show_missing_data(column='Age')


In [ ]:
# Adjust column_names to match your dataset
column_names = ['team', 'targeted_productivity', 'smv', 'wip', 'over_time',
                'incentive', 'idle_time', 'idle_men', 'no_of_style_change',
                'no_of_workers', 'actual_productivity', 'count']
# column_names = ['Voltage (V)', 'Current (A)', 'Motor Speed (RPM)', 'Temperature (°C)']
inspector.plot_numerical(column_names=column_names)


In [ ]:
inspector.handle_outliers(
    columns=['smv', 'incentive', 'targeted_productivity'],
    find_and_delete=True
)
# Or preview only: inspector.handle_outliers(find_and_delete=False)


In [ ]:
inspector.plot_categorical(column_names=['date', 'quarter', 'department', 'day'])


In [ ]:
# Update column names to match your dataset
inspector.plot_relationship('What was your Z-score at A/L', 'What is your GPA after 1st semester ?')


In [ ]:
# Delete rows by index — pass a list of integer indices to drop.
# Run with no arguments first to see the current index range.
inspector.delete_rows()
# Example: inspector.delete_rows([0, 5, 12])


In [ ]:
# Pass a controlled subset of highly variable features
features_to_test = column_names
# features_to_test = ['Temperature (°C)', 'Humidity (%)', 'CO2 (ppm)']

# Run the mean test on these specific features
inspector.test_constant_mean(columns=features_to_test, chunks=10)


## Initial Data Association Plots

In [ ]:
inspector.plot_numerical_correlation()


In [ ]:
inspector.plot_categorical_correlation()


In [ ]:
inspector.plot_all_associations_heatmap()


## Testing for Weak iid

In [ ]:
inspector.test_constant_covariance(columns=column_names)


In [ ]:
inspector.test_row_independence(columns=column_names)


In [ ]:
inspector.instantiate_macro_clt_distribution(columns=column_names)


In [ ]:
inspector.estimate_joint_normal(columns=column_names)


## PCA

In [ ]:
inspector.compute_empirical_pca(columns=column_names)


## Factor Analysis

In [ ]:
inspector.compute_empirical_fa(columns=column_names, k=3)


## Export Cleaned Data

Saves the current DataFrame as a CSV and, in Colab, automatically triggers a browser download.

In [ ]:
# Save and download the cleaned dataset
inspector.export_cleaned_data(filename='cleaned_data.csv')


## Custom Plotting

The `PlottingMethods` class provides direct access to individual chart types. All methods return a result dict; use `PLT.display_image(result)` to render in Colab.

In [ ]:
# Create PlottingMethods instance
PLT = PlottingMethods()


In [ ]:
# Display all available plotting methods
response = PLT.get_methods_info()
pd.DataFrame(response.get('response', {}))


In [ ]:
inspector.df.columns


In [ ]:
result = PLT.plot_pie_chart(names='team', values='incentive', data=inspector.df)
print(result.get('status'))
PLT.display_image(result=result)


In [ ]:
result = PLT.plot_bar_chart(x='team', y='incentive', data=inspector.df)
PLT.display_image(result=result)


In [ ]:
result = PLT.plot_histogram(x='incentive', data=inspector.df)
PLT.display_image(result=result)


In [ ]:
title = 'Productivity'
result = PLT.plot_heat_map(
    values='actual_productivity',
    index='department',
    columns='quarter',
    aggregade_method='mean',
    title=title,
    data=inspector.df
)
PLT.display_image(result=result)


In [ ]:
result = PLT.plot_sankey_diagram(
    source_column='department',
    target_column='quarter',
    values='actual_productivity',
    data=inspector.df
)
print(result.get('status'))
PLT.display_image(result=result)


In [ ]:
title = 'Maths and Physics vs Expected Destination'
result = PLT.plot_simple_sunburst_graph(
    path=['department', 'quarter'],
    values='actual_productivity',
    data=inspector.df,
    title=title
)
print(result.get('status'))
PLT.display_image(result=result)
